# iTRAILS results

Loads the outputs produced by the workflow (`gwf run` in the repository root) and
gives a first look at each analysis: the fitted model parameters, the Viterbi
segmentation of the alignment into gene-tree states, and the posterior state
probabilities along the alignment.

The notebook is executed by the workflow itself once all upstream targets are
done, so it simply discovers whatever analyses exist under `steps/itrails/`
(each analysis has one subfolder per step: `split/`, `optimize/`, `viterbi/`,
`posterior/`, `concat/`).

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import display, Markdown

sns.set_style('whitegrid')
%config InlineBackend.figure_formats = ['retina']

Discover the analyses that have finished (the notebook runs with `notebooks/` as
working directory, so results sit one level up):

In [ ]:
steps_dir = Path('..') / 'steps' / 'itrails'
analyses = sorted(p for p in steps_dir.glob('*') if p.is_dir()) if steps_dir.exists() else []
if not analyses:
    print('No iTRAILS results found yet - run the workflow first (see the README).')
[p.name for p in analyses]

## Fitted model parameters

The best parameter set found by `itrails-optimize` for each analysis
(times in generations, `r` per site per generation):

In [ ]:
for analysis in analyses:
    name = analysis.name
    best_model = analysis / 'optimize' / f'{name}.best_model.yaml'
    if not best_model.exists():
        # fit: window mode has one best_model per window instead
        window_models = sorted((analysis / 'optimize').glob('*.best_model.yaml'))
        display(Markdown(f"### {name}\n{len(window_models)} per-window model fits "
                         f"in `{analysis / 'optimize'}`"))
        continue
    with open(best_model) as f:
        best = yaml.safe_load(f)
    display(Markdown(f"### {name}\n"
                     f"log likelihood: **{best['results']['log_likelihood']}** "
                     f"(iteration {best['results']['iteration']})"))
    display(pd.Series(best['optimized_parameters'], name='estimate', dtype=float).to_frame())

## Viterbi decoding

How much of the alignment is assigned to each hidden state (a gene-tree topology
combined with coalescence-time intervals), and the state sequence along the
genome (windowed analyses, using `concat/{name}.viterbi.csv`) or along the
largest alignment block (unwindowed analyses):

In [ ]:
for analysis in analyses:
    name = analysis.name
    windowed = (analysis / 'concat' / f'{name}.viterbi.csv').exists()
    if windowed:
        # concatenated per-window decode; block_start + block-local position
        # places each segment on the binning-reference genome
        vit = pd.read_csv(analysis / 'concat' / f'{name}.viterbi.csv')
        vit['genome_start'] = vit['block_start'] + vit['position_start']
        vit['genome_end'] = vit['block_start'] + vit['position_end']
        # under fit: genome every window has the same state legend
        states_file = sorted((analysis / 'viterbi').glob('*.hidden_states.csv'))[0]
    else:
        vit = pd.read_csv(analysis / 'viterbi' / f'{name}.viterbi.csv')
        states_file = analysis / 'viterbi' / f'{name}.hidden_states.csv'
    states = pd.read_csv(states_file)
    labels = states.set_index('state_idx')['shorthand_name']

    vit['length'] = vit['position_end'] - vit['position_start'] + 1
    bp_per_state = (vit.groupby('most_likely_state')['length'].sum()
                       .rename(index=labels).sort_values())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    bp_per_state.plot.barh(ax=axes[0], color='C0')
    axes[0].set_xlabel('bases assigned')
    axes[0].set_ylabel('hidden state')

    if windowed:
        for seg in vit.itertuples():
            axes[1].hlines(seg.most_likely_state, seg.genome_start, seg.genome_end,
                           lw=3, color='C0')
        axes[1].set_xlabel(f"position on {vit['chrom'].iloc[0]}")
    else:
        largest_block = vit.groupby('Block_idx')['length'].sum().idxmax()
        block = vit[vit['Block_idx'] == largest_block]
        for seg in block.itertuples():
            axes[1].hlines(seg.most_likely_state, seg.position_start, seg.position_end,
                           lw=3, color='C0')
        axes[1].set_xlabel(f'position in block {largest_block}')
    axes[1].set_ylabel('state index')

    fig.suptitle(f'{name}: Viterbi decoding')
    fig.tight_layout()
    plt.show()

## Posterior decoding

Posterior probability of every hidden state along the genome (windowed analyses,
using `concat/{name}.posterior.csv`) or along the largest alignment block
(unwindowed analyses). Positions are subsampled for plotting when the sequence
is long:

In [ ]:
for analysis in analyses:
    name = analysis.name
    windowed = (analysis / 'concat' / f'{name}.posterior.csv').exists()
    if windowed:
        post = pd.read_csv(analysis / 'concat' / f'{name}.posterior.csv')
        prob_cols = [c for c in post.columns if c.startswith('prob_state_')]
        post['genome_pos'] = post['block_start'] + post['position_idx']
        post = post.sort_values('genome_pos')
        probs = post[prob_cols].to_numpy().T
        xlabel = f"position on {post['chrom'].iloc[0]}"
    else:
        post = pd.read_csv(analysis / 'posterior' / f'{name}.posterior.csv')
        prob_cols = [c for c in post.columns if c.startswith('prob_state_')]
        block_col = post.columns[0]
        largest_block = post[block_col].value_counts().idxmax()
        probs = post[post[block_col] == largest_block][prob_cols].to_numpy().T
        xlabel = f'position in block {largest_block}'

    step = max(1, probs.shape[1] // 2000)
    plt.figure(figsize=(12, 4))
    plt.imshow(probs[:, ::step], aspect='auto', origin='lower',
               cmap='viridis', vmin=0, vmax=1, interpolation='nearest')
    plt.colorbar(label='posterior probability')
    plt.xlabel(xlabel + (f' (every {step} bp)' if step > 1 else ''))
    plt.ylabel('state index')
    plt.title(f'{name}: posterior decoding')
    plt.show()